In [31]:
import torch
import platform
import subprocess
import os

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

# Basic system info
print(f"Hostname: {platform.node()}")
print(f"OS: {platform.system()} {platform.release()}")
print(f"Python: {platform.python_version()}")
print(f"Python Executable: {os.sys.executable}")

print("\n" + "=" * 60)
print("CPU INFORMATION")
print("=" * 60)

# CPU info
try:
    cpu_info = subprocess.check_output("lscpu | grep 'Model name'", shell=True).decode()
    print(cpu_info.strip())
    cpu_count = os.cpu_count()
    print(f"CPU Cores Available: {cpu_count}")
except:
    print(f"CPU Cores: {os.cpu_count()}")

print("\n" + "=" * 60)
print("MEMORY INFORMATION")
print("=" * 60)

# Memory info
try:
    mem_info = subprocess.check_output("free -h | grep Mem", shell=True).decode()
    parts = mem_info.split()
    print(f"Total RAM: {parts[1]}")
    print(f"Used RAM: {parts[2]}")
    print(f"Available RAM: {parts[6]}")
except:
    print("Memory info unavailable")

print("\n" + "=" * 60)
print("GPU INFORMATION")
print("=" * 60)

# PyTorch CUDA info
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    # Detailed info for each GPU
    for i in range(torch.cuda.device_count()):
        print(f"\n--- GPU {i} ---")
        print(f"Name: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"Compute Capability: {props.major}.{props.minor}")
        print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"Memory Allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
        print(f"Memory Reserved: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")
        print(f"Multi-Processors: {props.multi_processor_count}")
else:
    print("No CUDA GPUs detected")

print("\n" + "=" * 60)
print("STORAGE INFORMATION")
print("=" * 60)

# Disk space
try:
    disk_info = subprocess.check_output(f"df -h {os.path.expanduser('~')}", shell=True).decode()
    lines = disk_info.strip().split('\n')
    if len(lines) > 1:
        print(lines[0])  # Header
        print(lines[1])  # Home directory info
except:
    print("Disk info unavailable")

print("=" * 60)

SYSTEM INFORMATION
Hostname: nodegpu113.hpc.fau.edu
OS: Linux 4.18.0-553.58.1.el8_10.x86_64
Python: 3.13.11
Python Executable: /mnt/beegfs/home/yyu2024/my_pytorch_env/bin/python

CPU INFORMATION
Model name:          Intel(R) Xeon(R) Gold 6130 CPU @ 2.10GHz
CPU Cores Available: 64

MEMORY INFORMATION
Total RAM: 187Gi
Used RAM: 13Gi
Available RAM: 161Gi

GPU INFORMATION
PyTorch Version: 2.6.0+cu124
CUDA Available: True
CUDA Version: 12.4
cuDNN Version: 90100
Number of GPUs: 4

--- GPU 0 ---
Name: Tesla V100-SXM2-32GB
Compute Capability: 7.0
Total Memory: 31.73 GB
Memory Allocated: 0.00 GB
Memory Reserved: 0.03 GB
Multi-Processors: 80

--- GPU 1 ---
Name: Tesla V100-SXM2-32GB
Compute Capability: 7.0
Total Memory: 31.73 GB
Memory Allocated: 0.00 GB
Memory Reserved: 0.00 GB
Multi-Processors: 80

--- GPU 2 ---
Name: Tesla V100-SXM2-32GB
Compute Capability: 7.0
Total Memory: 31.73 GB
Memory Allocated: 0.00 GB
Memory Reserved: 0.00 GB
Multi-Processors: 80

--- GPU 3 ---
Name: Tesla V100-SXM2-3

In [32]:
import sys
print(sys.executable)

!{sys.executable} -m pip install numpy mne scipy plotly pandas scikit-learn pytorch-model-summary wandb


/mnt/beegfs/home/yyu2024/my_pytorch_env/bin/python


In [ ]:
import numpy as np
import scipy.io, scipy.interpolate
import pathlib
import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl
from pytorch_model_summary import summary
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import scipy.ndimage
import plotly.tools as tls
from pytorch_lightning.callbacks import Callback
from pytorch_lightning import Trainer
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

L_FREQ, H_FREQ = 40, 300
CHANNELS_NUM = 62
WAVELET_NUM = 40
DOWNSAMPLE_FS = 100
time_delay_secs = 0.2
current_fs = DOWNSAMPLE_FS

# =============================================================================
# MODEL SELECTION
# =============================================================================
# 'fingerflex'     - Original FingerFlex U-Net with wavelet spectrograms
# 'bc4d4'          - BC4D4 single time point (doesn't work well)
# 'bc4d4_windowed' - BC4D4 with temporal windows (re-interpreted from paper)
MODEL_MODE = 'bc4d4_windowed'  # <-- USE THIS FOR WINDOWED VERSION

# =============================================================================
# BC4D4 WINDOWED SETTINGS
# =============================================================================
BC4D4_WINDOW_SIZE = 7         # Temporal window (7 matches paper's param count)
BC4D4_ACTIVATION = 'softsign' # 'tanh' or 'softsign'
BC4D4_FINGER_IDX = 0          # 0=Thumb, 1=Index, 2=Middle, 3=Ring, 4=Little

FINGER_NAMES = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']

TYPE = "train"
model_to_test = f"{pathlib.Path().resolve()}/checkpoints/model-epoch=16-corr_mean_val=0.6790410876274109.ckpt"

print(f"Model mode: {MODEL_MODE}")
if MODEL_MODE == 'bc4d4_windowed':
    print(f"BC4D4 Windowed Settings:")
    print(f"  Window size: {BC4D4_WINDOW_SIZE} time steps")
    print(f"  Activation: {BC4D4_ACTIVATION}")
    if BC4D4_FINGER_IDX is not None:
        print(f"  Finger: {FINGER_NAMES[BC4D4_FINGER_IDX]} (index {BC4D4_FINGER_IDX})")
    else:
        print(f"  Predicting all 5 fingers")
elif MODEL_MODE in ['bc4d4', 'bc4d4_iso']:
    print(f"BC4D4 single time point mode")
    if BC4D4_FINGER_IDX is not None:
        print(f"  Finger: {FINGER_NAMES[BC4D4_FINGER_IDX]}")
print(f"Run type: {TYPE}")

In [ ]:
# =============================================================================
# DATASET CLASSES
# =============================================================================

class EcogFingerflexDataset(Dataset):
    """Dataset for FingerFlex model (wavelet spectrograms)"""
    def __init__(self, path_to_ecog_data: str,
                 path_to_fingerflex_data: str, sample_len: int, train = False):
        self.ecog_data, self.fingerflex_data = np.load(path_to_ecog_data).astype('float32'),\
                                            np.load(path_to_fingerflex_data).astype('float32')
        
        self.duration = self.ecog_data.shape[2]
        self.sample_len = sample_len
        self.stride = 1
        self.ds_len = (self.duration-self.sample_len) // self.stride
        self.train = train
        
        print("Duration: ", self.duration, "Ds_len:", self.ds_len)
        
    def __len__(self):
        return self.ds_len
    
    def __getitem__(self, index):
        sample_start = index*self.stride
        sample_end = sample_start+self.sample_len
        ecog_sample = self.ecog_data[...,sample_start:sample_end]
        fingerflex_sample = self.fingerflex_data[...,sample_start:sample_end]
        return ecog_sample, fingerflex_sample


class BC4D4Dataset(Dataset):
    """
    Dataset for BC4D4 model (raw ECoG signals) - single time point version
    ECoG shape: (time, electrodes, 1)
    Finger shape: (time, 5)
    """
    def __init__(self, path_to_ecog_data: str,
                 path_to_fingerflex_data: str, 
                 finger_idx: int = None,
                 train: bool = False):
        self.ecog_data = np.load(path_to_ecog_data).astype('float32')
        self.fingerflex_data = np.load(path_to_fingerflex_data).astype('float32')
        self.finger_idx = finger_idx
        self.train = train
        self.n_samples = self.ecog_data.shape[0]
        
        print(f"BC4D4 Dataset - Samples: {self.n_samples}, "
              f"Electrodes: {self.ecog_data.shape[1]}, "
              f"Finger idx: {finger_idx}")
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, index):
        ecog_sample = self.ecog_data[index]
        if self.finger_idx is not None:
            finger_sample = self.fingerflex_data[index, self.finger_idx:self.finger_idx+1]
        else:
            finger_sample = self.fingerflex_data[index]
        return ecog_sample, finger_sample


class BC4D4WindowedDataset(Dataset):
    """
    Dataset for BC4D4Windowed model - uses temporal windows
    
    Creates sliding windows over the ECoG data.
    Each sample is (window_size, num_electrodes) and predicts the finger
    position at the END of the window.
    
    Parameters
    ----------
    path_to_ecog_data : str
        Path to ECoG data file. Expected shape: (time, electrodes) or (time, electrodes, 1)
    path_to_fingerflex_data : str
        Path to finger data file. Expected shape: (time, 5)
    window_size : int
        Number of time steps in each window (default: 7)
    finger_idx : int or None
        Which finger to predict (0-4), or None for all 5
    stride : int
        Step size between consecutive windows (default: 1)
    """
    def __init__(self, path_to_ecog_data: str,
                 path_to_fingerflex_data: str,
                 window_size: int = 7,
                 finger_idx: int = None,
                 stride: int = 1,
                 train: bool = False):
        
        ecog_raw = np.load(path_to_ecog_data).astype('float32')
        self.fingerflex_data = np.load(path_to_fingerflex_data).astype('float32')
        
        # Handle different input shapes
        if ecog_raw.ndim == 3 and ecog_raw.shape[2] == 1:
            # Shape: (time, electrodes, 1) -> (time, electrodes)
            self.ecog_data = ecog_raw.squeeze(-1)
        elif ecog_raw.ndim == 2:
            # Shape: (time, electrodes) - already correct
            self.ecog_data = ecog_raw
        else:
            raise ValueError(f"Unexpected ECoG shape: {ecog_raw.shape}")
        
        self.window_size = window_size
        self.finger_idx = finger_idx
        self.stride = stride
        self.train = train
        
        self.n_timepoints = self.ecog_data.shape[0]
        self.num_electrodes = self.ecog_data.shape[1]
        
        # Number of valid windows
        self.n_samples = (self.n_timepoints - window_size) // stride + 1
        
        print(f"BC4D4Windowed Dataset:")
        print(f"  Total timepoints: {self.n_timepoints}")
        print(f"  Electrodes: {self.num_electrodes}")
        print(f"  Window size: {window_size}")
        print(f"  Stride: {stride}")
        print(f"  Valid samples: {self.n_samples}")
        print(f"  Finger idx: {finger_idx}")
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, index):
        # Calculate start position
        start = index * self.stride
        end = start + self.window_size
        
        # Extract window: (window_size, num_electrodes)
        ecog_window = self.ecog_data[start:end, :]
        
        # Target is finger position at END of window
        target_idx = end - 1
        
        if self.finger_idx is not None:
            finger_target = self.fingerflex_data[target_idx, self.finger_idx:self.finger_idx+1]
        else:
            finger_target = self.fingerflex_data[target_idx, :]
        
        return ecog_window, finger_target


class EcogFingerflexDatamodule(pl.LightningDataModule):
    """
    DataModule supporting FingerFlex, BC4D4, and BC4D4Windowed models
    """
    def __init__(self, sample_len: int, data_dir = f"{pathlib.Path().resolve()}/data",
                    batch_size=128, add_name="", model_mode='fingerflex', 
                    finger_idx=None, window_size=7, stride=1):
        super().__init__()
        self.data_dir = data_dir
        self.sample_len = sample_len
        self.batch_size = batch_size
        self.add_name = add_name
        self.model_mode = model_mode
        self.finger_idx = finger_idx
        self.window_size = window_size
        self.stride = stride
        
    def setup(self, stage = None):
        if self.model_mode == 'fingerflex':
            if stage is None or stage == "fit":
                self.train = EcogFingerflexDataset(f"{self.data_dir}/train/ecog_data{self.add_name}.npy",
                                                  f"{self.data_dir}/train/fingerflex_data{self.add_name}.npy",
                                                  self.sample_len, train = True)
                self.val = EcogFingerflexDataset(f"{self.data_dir}/val/ecog_data{self.add_name}.npy",
                                                  f"{self.data_dir}/val/fingerflex_data{self.add_name}.npy",
                                                  self.sample_len)
            if stage is None or stage == "test":
                self.test = EcogFingerflexDataset(f"{self.data_dir}/test/ecog_data{self.add_name}.npy",
                                                  f"{self.data_dir}/test/fingerflex_data{self.add_name}.npy",
                                                  self.sample_len)
        
        elif self.model_mode == 'bc4d4':
            # Original BC4D4 with single time points
            if stage is None or stage == "fit":
                self.train = BC4D4Dataset(f"{self.data_dir}/train/ecog_data{self.add_name}.npy",
                                          f"{self.data_dir}/train/fingerflex_data{self.add_name}.npy",
                                          finger_idx=self.finger_idx, train=True)
                self.val = BC4D4Dataset(f"{self.data_dir}/val/ecog_data{self.add_name}.npy",
                                        f"{self.data_dir}/val/fingerflex_data{self.add_name}.npy",
                                        finger_idx=self.finger_idx)
            if stage is None or stage == "test":
                self.test = BC4D4Dataset(f"{self.data_dir}/test/ecog_data{self.add_name}.npy",
                                         f"{self.data_dir}/test/fingerflex_data{self.add_name}.npy",
                                         finger_idx=self.finger_idx)
        
        elif self.model_mode == 'bc4d4_windowed':
            # New windowed BC4D4
            if stage is None or stage == "fit":
                self.train = BC4D4WindowedDataset(
                    f"{self.data_dir}/train/ecog_data{self.add_name}.npy",
                    f"{self.data_dir}/train/fingerflex_data{self.add_name}.npy",
                    window_size=self.window_size,
                    finger_idx=self.finger_idx,
                    stride=self.stride,
                    train=True
                )
                self.val = BC4D4WindowedDataset(
                    f"{self.data_dir}/val/ecog_data{self.add_name}.npy",
                    f"{self.data_dir}/val/fingerflex_data{self.add_name}.npy",
                    window_size=self.window_size,
                    finger_idx=self.finger_idx,
                    stride=self.stride
                )
            if stage is None or stage == "test":
                self.test = BC4D4WindowedDataset(
                    f"{self.data_dir}/test/ecog_data{self.add_name}.npy",
                    f"{self.data_dir}/test/fingerflex_data{self.add_name}.npy",
                    window_size=self.window_size,
                    finger_idx=self.finger_idx,
                    stride=self.stride
                )
    
    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, num_workers=4, shuffle=True)
    
    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.batch_size)
    
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.batch_size)


print("Dataset classes loaded: EcogFingerflexDataset, BC4D4Dataset, BC4D4WindowedDataset")

In [ ]:
def correlation_metric(x, y):
    """
     Cosine distance calculation metric
    """
    cos_metric = nn.CosineSimilarity(dim=-1, eps=1e-08)

    cos_sim = torch.mean(cos_metric(x, y))

    return cos_sim

def corr_metric(x, y):
    """
    Pearson correlation calculation metric between univariate vectors
    """
    assert x.shape == y.shape  
    r = np.corrcoef(x, y)[0, 1]
    return r


class BaseEcogFingerflexModel(pl.LightningModule):
    """
    Lightning wrapper for FingerFlex model
    Uses cosine similarity + MSE loss
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.lr = 8.42e-5
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self.model(x)
        
        loss = F.mse_loss(y_hat, y)
        corr = correlation_metric(y_hat, y)

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log(f"cosine_dst_train", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        return 0.5*loss + 0.5*(1. - corr)
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.mse_loss(y_hat, y)
        
        corr = correlation_metric(y_hat, y)

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("cosine_dst_val", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        
        return y_hat
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self.model(x)
        
        loss = F.mse_loss(y_hat, y)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-6)
        return optimizer


class BC4D4LightningModel(pl.LightningModule):
    """
    Lightning wrapper for BC4D4 model
    Uses MSE loss (regression task)
    """
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self.model(x)
        
        loss = F.mse_loss(y_hat, y)
        
        # Calculate Pearson correlation for monitoring
        with torch.no_grad():
            y_np = y.cpu().numpy().flatten()
            y_hat_np = y_hat.cpu().numpy().flatten()
            corr = np.corrcoef(y_np, y_hat_np)[0, 1] if len(y_np) > 1 else 0

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_corr", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.mse_loss(y_hat, y)
        
        # Calculate Pearson correlation
        with torch.no_grad():
            y_np = y.cpu().numpy().flatten()
            y_hat_np = y_hat.cpu().numpy().flatten()
            corr = np.corrcoef(y_np, y_hat_np)[0, 1] if len(y_np) > 1 else 0

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_corr", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        
        return y_hat
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.mse_loss(y_hat, y)
        
        with torch.no_grad():
            y_np = y.cpu().numpy().flatten()
            y_hat_np = y_hat.cpu().numpy().flatten()
            corr = np.corrcoef(y_np, y_hat_np)[0, 1] if len(y_np) > 1 else 0
        
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("test_corr", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=0)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10, verbose=True
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }


print("Lightning wrappers loaded: BaseEcogFingerflexModel, BC4D4LightningModel")

In [ ]:
"""
Model architectures: FingerFlex (U-Net) and BC4D4 (CNN+DNN)
"""

# =============================================================================
# FINGERFLEX MODEL (Original U-Net Architecture)
# =============================================================================

class ConvBlock(nn.Module):
    """
    Convolution block:
        - 1d conv
        - layer norm by embedding axis
        - activation
        - dropout
        - Max pooling
    """
    def __init__(self, in_channels, out_channels, kernel_size, 
                 stride=1, dilation=1, p_conv_drop=0.1):
        super(ConvBlock, self).__init__()
        
        self.conv1d = nn.Conv1d(in_channels, out_channels, 
                                kernel_size=kernel_size, 
                                bias=False, 
                                padding='same')
        
        self.norm = nn.LayerNorm(out_channels)
        self.activation = nn.GELU()
        self.drop = nn.Dropout(p=p_conv_drop)
        self.downsample = nn.MaxPool1d(kernel_size=stride, stride=stride)

        self.stride = stride
        self.in_channels = in_channels
        self.out_channels = out_channels
        
    def forward(self, x):
        x = self.conv1d(x)
        x = torch.transpose(x, -2, -1) 
        x = self.norm(x)
        x = torch.transpose(x, -2, -1)
        x = self.activation(x)
        x = self.drop(x)
        x = self.downsample(x)
        return x


class UpConvBlock(nn.Module):
    """Decoder convolution block"""
    def __init__(self, scale, **args):
        super(UpConvBlock, self).__init__()
        self.conv_block = ConvBlock(**args)
        self.upsample = nn.Upsample(scale_factor=scale, mode='linear', align_corners=False)

    def forward(self, x):
        x = self.conv_block(x)
        x = self.upsample(x)
        return x    


class AutoEncoder1D(nn.Module):
    """FingerFlex Encoder-Decoder model with skip connections"""
    def __init__(self,
                 n_electrodes=30,
                 n_freqs = 16,
                 n_channels_out=21,
                 channels = [8, 16, 32, 32],
                 kernel_sizes=[3, 3, 3],
                 strides=[4, 4, 4],
                 dilation=[1, 1, 1]
                 ):
        super(AutoEncoder1D, self).__init__()
        
        self.n_electrodes = n_electrodes
        self.n_freqs = n_freqs
        self.n_inp_features = n_freqs*n_electrodes
        self.n_channels_out = n_channels_out
        
        self.model_depth = len(channels)-1
        self.spatial_reduce = ConvBlock(self.n_inp_features, channels[0], kernel_size=3)
        
        self.downsample_blocks = nn.ModuleList([ConvBlock(channels[i], 
                                                        channels[i+1], 
                                                        kernel_sizes[i],
                                                        stride=strides[i], 
                                                        dilation=dilation[i]) for i in range(self.model_depth)])

        channels = [ch for ch in channels[:-1]] + channels[-1:]

        self.upsample_blocks = nn.ModuleList([UpConvBlock(scale=strides[i],
                                                          in_channels=channels[i+1] if i == self.model_depth-1 else channels[i+1]*2,
                                                          out_channels=channels[i],
                                                          kernel_size=kernel_sizes[i]) for i in range(self.model_depth-1, -1, -1)])
        
        self.conv1x1_one = nn.Conv1d(channels[0]*2, self.n_channels_out, kernel_size=1, padding='same')
      
    def forward(self, x):
        batch, elec, n_freq, time = x.shape
        x = x.reshape(batch, -1, time)
        x = self.spatial_reduce(x)
        
        skip_connection = []
        for i in range(self.model_depth):
            skip_connection.append(x)
            x = self.downsample_blocks[i](x)

        for i in range(self.model_depth):
            x = self.upsample_blocks[i](x)
            x = torch.cat((x, skip_connection[-1 - i]), dim=1)
        
        x = self.conv1x1_one(x)
        return x


# =============================================================================
# BC4D4 MODEL WITH TEMPORAL WINDOWS (Re-interpreted from paper)
# =============================================================================
# Based on reverse-engineering the paper's parameter counts:
# - Table 3 shows Dense layer params = 262,144 = 256 × 1024
# - This means flatten output = 256, not 15872
# - With padding='valid' and 3 Conv layers (kernel=3), dimension reduces by 6
# - So input window = 7 time steps → output = 1 → flatten = 256
#
# Architecture:
#   Input: (batch, window_size, num_electrodes) e.g., (batch, 7, 62)
#   Conv1D (valid): window_size → window_size-2
#   Conv1D (valid): → window_size-4
#   Conv1D (valid): → window_size-6 = 1
#   Flatten: 1 × 256 = 256
#   Dense layers: 256 → 1024 → dropout → 512 → 256 → 128 → 64 → 1

class Softsign(nn.Module):
    """Softsign activation: f(x) = x / (1 + |x|)"""
    def forward(self, x):
        return x / (1 + torch.abs(x))


class BC4D4Windowed(nn.Module):
    """
    BC4D4 model with temporal windows - re-interpreted from paper.
    
    The paper's Table 3 parameter counts only make sense if:
    - Input has a small temporal window (7 time steps)
    - Conv1D uses padding='valid' (reduces dimension)
    - Flatten output = 256 (matches 256×1024 = 262,144 params)
    
    Parameters
    ----------
    num_electrodes : int
        Number of ECoG electrodes (62 for Subject 1)
    window_size : int
        Temporal window size (default: 7 to match paper's param counts)
    activation : str
        'tanh' or 'softsign'
    dropout_rate : float
        Dropout rate (paper uses 0.1)
    n_outputs : int
        Number of outputs (1 for single finger, 5 for all)
    """
    def __init__(self, num_electrodes: int = 62, window_size: int = 7,
                 activation: str = 'softsign', dropout_rate: float = 0.1, 
                 n_outputs: int = 1):
        super(BC4D4Windowed, self).__init__()
        
        self.num_electrodes = num_electrodes
        self.window_size = window_size
        self.activation_name = activation
        self.n_outputs = n_outputs
        
        # CNN Block with padding='valid' (reduces spatial dimension)
        # Input: (batch, 1, window_size) - treating electrodes as batch/channels
        # Actually: (batch, num_electrodes, window_size) then conv over time
        self.conv1 = nn.Conv1d(num_electrodes, 64, kernel_size=3, stride=1, padding=0)  # valid padding
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=0)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=0)
        
        # After 3 conv layers with kernel=3, valid padding: window_size - 6
        # For window_size=7: 7-6=1, so output is (batch, 256, 1)
        self.temporal_output = window_size - 6
        assert self.temporal_output >= 1, f"Window size {window_size} too small, need at least 7"
        
        # Flatten size: temporal_output × 256 channels
        self.flatten_size = self.temporal_output * 256
        
        # Select activation
        if activation == 'tanh':
            self.dense_activation = nn.Tanh()
        elif activation == 'softsign':
            self.dense_activation = Softsign()
        else:
            raise ValueError(f"Unknown activation: {activation}")
        
        # DNN Block (from Table 3)
        self.fc1 = nn.Linear(self.flatten_size, 1024)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64)
        self.fc6 = nn.Linear(64, n_outputs)
    
    def forward(self, x):
        """
        Forward pass.
        
        Input shape: (batch, window_size, num_electrodes) e.g., (batch, 7, 62)
        Output shape: (batch, n_outputs)
        """
        # Permute to (batch, num_electrodes, window_size) for Conv1d
        # Conv1d expects (batch, channels, length)
        x = x.permute(0, 2, 1)  # (batch, 62, 7)
        
        # CNN Block with ReLU and valid padding
        x = F.relu(self.conv1(x))  # (batch, 64, 5)
        x = F.relu(self.conv2(x))  # (batch, 128, 3)
        x = F.relu(self.conv3(x))  # (batch, 256, 1)
        
        # Flatten
        x = x.view(x.size(0), -1)  # (batch, 256)
        
        # DNN Block with Tanh/Softsign
        x = self.dense_activation(self.fc1(x))
        x = self.dropout(x)
        x = self.dense_activation(self.fc2(x))
        x = self.dense_activation(self.fc3(x))
        x = self.dense_activation(self.fc4(x))
        x = self.dense_activation(self.fc5(x))
        x = self.dense_activation(self.fc6(x))
        
        return x
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Keep the old BC4D4 class for backwards compatibility
class BC4D4(nn.Module):
    """Original BC4D4 (single time point) - kept for reference"""
    def __init__(self, num_features: int, activation: str = 'softsign',
                 dropout_rate: float = 0.1, n_outputs: int = 1):
        super(BC4D4, self).__init__()
        
        self.num_features = num_features
        self.activation_name = activation
        self.n_outputs = n_outputs
        
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, stride=1, padding='same')
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, stride=1, padding='same')
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, stride=1, padding='same')
        
        self.flatten_size = num_features * 256
        
        if activation == 'tanh':
            self.dense_activation = nn.Tanh()
        elif activation == 'softsign':
            self.dense_activation = Softsign()
        else:
            raise ValueError(f"Unknown activation: {activation}")
        
        self.fc1 = nn.Linear(self.flatten_size, 1024)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64)
        self.fc6 = nn.Linear(64, n_outputs)
    
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = self.dense_activation(self.fc1(x))
        x = self.dropout(x)
        x = self.dense_activation(self.fc2(x))
        x = self.dense_activation(self.fc3(x))
        x = self.dense_activation(self.fc4(x))
        x = self.dense_activation(self.fc5(x))
        x = self.dense_activation(self.fc6(x))
        return x
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("Models loaded: AutoEncoder1D (FingerFlex), BC4D4, BC4D4Windowed")

In [ ]:
class ValidationCallback(Callback):
    """
    Callback for FingerFlex model - calculates correlation at end of each validation epoch
    """
    def __init__(self, val_x, val_y, fg_num):
        super().__init__()
        self.val_x = val_x.T
        self.val_y = val_y.T
        self.fg_num = fg_num

    def on_validation_epoch_end(self, trainer, pl_module):
        with torch.no_grad():
            SIZE = 64
            bound = self.val_x.shape[0]//SIZE *SIZE

            X_test = self.val_x[:bound]
            y_test = self.val_y[:bound]
            x_batch = torch.from_numpy(X_test).float().to("cuda:0")

            x_batch = x_batch.T
            x_batch = torch.unsqueeze(x_batch, 0)

            y_hat = pl_module.model(x_batch)[0]
            y_hat = y_hat.cpu().detach().numpy()
            STRIDE = 1
            y_prediction = y_hat.T[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]
            y_prediction = scipy.ndimage.gaussian_filter1d(y_prediction.T,sigma=6).T

            y_test = y_test[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]

            h, w = self.fg_num//2, self.fg_num - self.fg_num//2
            fig, ax = plt.subplots(h, w, figsize = (h*5, w*6), sharex=True, sharey=True)
            corrs = []

            for roi in range(self.fg_num):
                y_hat = y_prediction[:, roi]
                y_test_roi = y_test[:, roi]
                corr_tmp = corr_metric(y_hat, y_test_roi)
                corrs.append(corr_tmp)
                axi = ax.flat[roi]
                axi.plot(y_hat, label= 'prediction')
                axi.plot(y_test_roi, label = 'true')
                axi.set_title("RoI {}_corr {:.2f}".format(roi, corr_tmp))

            corr_mean = np.mean(corrs)
            pl_module.log("corr_mean_val", corr_mean, on_step=False, on_epoch=True, prog_bar=True, logger=True)
            wandb.log({f"plots": fig})
            plt.close(fig)


class BC4D4ValidationCallback(Callback):
    """
    Callback for BC4D4 model - calculates Pearson correlation at end of each validation epoch.
    Supports both single time point and windowed BC4D4 models.
    """
    def __init__(self, val_x, val_y, finger_idx=None, window_size=None):
        super().__init__()
        self.val_x = val_x.astype('float32')  # Shape: (time, electrodes) or (time, electrodes, 1)
        self.val_y = val_y.astype('float32')  # Shape: (time, 5)
        self.finger_idx = finger_idx
        self.window_size = window_size  # If set, use windowed mode
        self.finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']
        
        # Handle different input shapes - squeeze if needed
        if self.val_x.ndim == 3 and self.val_x.shape[2] == 1:
            self.val_x = self.val_x.squeeze(-1)  # (time, electrodes)

    def on_validation_epoch_end(self, trainer, pl_module):
        with torch.no_grad():
            batch_size = 512
            
            # Check if model is windowed by looking at model type
            is_windowed = hasattr(pl_module.model, 'window_size')
            
            if is_windowed:
                # Windowed mode: create sliding windows
                window_size = pl_module.model.window_size
                n_timepoints = len(self.val_x)
                n_samples = n_timepoints - window_size + 1
                
                all_preds = []
                
                for i in range(0, n_samples, batch_size):
                    # Create batch of windows
                    batch_end = min(i + batch_size, n_samples)
                    windows = []
                    for j in range(i, batch_end):
                        window = self.val_x[j:j+window_size, :]  # (window_size, electrodes)
                        windows.append(window)
                    
                    batch = torch.from_numpy(np.array(windows)).float().to("cuda:0")
                    pred = pl_module.model(batch)
                    all_preds.append(pred.cpu().numpy())
                
                y_pred = np.concatenate(all_preds, axis=0)
                
                # Targets correspond to end of each window
                y_true_all = self.val_y[window_size-1:window_size-1+n_samples]
            else:
                # Single time point mode
                n_samples = len(self.val_x)
                
                all_preds = []
                x_tensor = torch.from_numpy(self.val_x).float().to("cuda:0")
                
                for i in range(0, n_samples, batch_size):
                    batch = x_tensor[i:i+batch_size]
                    pred = pl_module.model(batch)
                    all_preds.append(pred.cpu().numpy())
                
                y_pred = np.concatenate(all_preds, axis=0)
                y_true_all = self.val_y
            
            # Get the appropriate ground truth
            if self.finger_idx is not None:
                y_true = y_true_all[:, self.finger_idx]
                finger_name = self.finger_names[self.finger_idx]
                
                # Compute correlation
                corr = corr_metric(y_true.flatten(), y_pred.flatten())
                
                pl_module.log(f"val_corr_{finger_name}", corr, on_step=False, on_epoch=True, prog_bar=True)
                pl_module.log("corr_mean_val", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)
                
                # Create plot
                fig, ax = plt.subplots(figsize=(12, 4))
                n_plot = min(1000, len(y_true))
                ax.plot(y_true[:n_plot], label='True', alpha=0.7)
                ax.plot(y_pred[:n_plot].flatten(), label='Predicted', alpha=0.7)
                ax.set_title(f'{finger_name} - Correlation: {corr:.4f}')
                ax.legend()
                ax.set_xlabel('Sample')
                ax.set_ylabel('Finger Position')
            else:
                # All 5 fingers
                corrs = []
                fig, axes = plt.subplots(1, 5, figsize=(20, 4))
                
                for i, name in enumerate(self.finger_names):
                    corr = corr_metric(y_true_all[:, i], y_pred[:, i])
                    corrs.append(corr)
                    pl_module.log(f"val_corr_{name}", corr, on_step=False, on_epoch=True)
                    
                    n_plot = min(500, len(y_true_all))
                    axes[i].plot(y_true_all[:n_plot, i], label='True', alpha=0.7)
                    axes[i].plot(y_pred[:n_plot, i], label='Pred', alpha=0.7)
                    axes[i].set_title(f'{name}: {corr:.3f}')
                
                corr_mean = np.mean(corrs)
                pl_module.log("corr_mean_val", corr_mean, on_step=False, on_epoch=True, prog_bar=True, logger=True)
                fig.suptitle(f'Average Correlation: {corr_mean:.4f}')
            
            plt.tight_layout()
            wandb.log({"val_predictions": fig})
            plt.close(fig)


class TestCallback:
    """
    Callback for FingerFlex test evaluation
    """
    def __init__(self, val_x, val_y, fg_num):
        super().__init__()
        self.val_x = val_x.T
        self.val_y = val_y.T
        self.fg_num = fg_num

    def test(self, pl_module):
        with torch.no_grad():
            SIZE = 64
            bound = self.val_x.shape[0]//SIZE *SIZE

            X_test = self.val_x[:bound]
            y_test = self.val_y[:bound]
            x_batch = torch.from_numpy(X_test).float().to("cuda:0")

            x_batch = x_batch.T
            x_batch = torch.unsqueeze(x_batch, 0)

            y_hat = pl_module.model(x_batch)[0]
            y_hat = y_hat.cpu().detach().numpy()
            STRIDE = 1
            y_prediction = y_hat.T[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]
            y_prediction = scipy.ndimage.gaussian_filter1d(y_prediction.T,sigma=1).T

            y_test = y_test[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]

            np.save(f"{pathlib.Path().resolve()}/res_npy/prediction2.npy", y_prediction)
            np.save(f"{pathlib.Path().resolve()}/res_npy/true2.npy", y_test)

            h, w = self.fg_num//2, self.fg_num - self.fg_num//2
            fig, ax = plt.subplots(h, w, figsize = (h*35, w*6), sharex=True, sharey=True)
            corrs = []

            for roi in range(self.fg_num):
                y_hat = y_prediction[:, roi]
                y_test_roi = y_test[:, roi]
                corr_tmp = corr_metric(y_hat, y_test_roi)
                corrs.append(corr_tmp)
                axi = ax.flat[roi]
                axi.plot(y_hat, label= 'prediction')
                axi.plot(y_test_roi, label = 'true')
                axi.set_title("RoI {}_corr {:.2f}".format(roi, corr_tmp))

            corr_mean = np.mean(corrs)
            plotly_fig = tls.mpl_to_plotly(fig)
            print(f"Mean correlation: {corr_mean}")
            plotly_fig.write_html("res.html")


class BC4D4TestCallback:
    """
    Test callback for BC4D4 model - supports both single time point and windowed versions
    """
    def __init__(self, val_x, val_y, finger_idx=None):
        self.val_x = val_x.astype('float32')
        self.val_y = val_y.astype('float32')
        self.finger_idx = finger_idx
        self.finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']
        
        # Handle different input shapes - squeeze if needed
        if self.val_x.ndim == 3 and self.val_x.shape[2] == 1:
            self.val_x = self.val_x.squeeze(-1)

    def test(self, pl_module):
        with torch.no_grad():
            batch_size = 512
            
            # Check if model is windowed
            is_windowed = hasattr(pl_module.model, 'window_size')
            
            if is_windowed:
                window_size = pl_module.model.window_size
                n_timepoints = len(self.val_x)
                n_samples = n_timepoints - window_size + 1
                
                all_preds = []
                
                for i in range(0, n_samples, batch_size):
                    batch_end = min(i + batch_size, n_samples)
                    windows = []
                    for j in range(i, batch_end):
                        window = self.val_x[j:j+window_size, :]
                        windows.append(window)
                    
                    batch = torch.from_numpy(np.array(windows)).float().to("cuda:0")
                    pred = pl_module.model(batch)
                    all_preds.append(pred.cpu().numpy())
                
                y_pred = np.concatenate(all_preds, axis=0)
                y_true_all = self.val_y[window_size-1:window_size-1+n_samples]
            else:
                n_samples = len(self.val_x)
                
                all_preds = []
                x_tensor = torch.from_numpy(self.val_x).float().to("cuda:0")
                
                for i in range(0, n_samples, batch_size):
                    batch = x_tensor[i:i+batch_size]
                    pred = pl_module.model(batch)
                    all_preds.append(pred.cpu().numpy())
                
                y_pred = np.concatenate(all_preds, axis=0)
                y_true_all = self.val_y
            
            print("=" * 60)
            print("BC4D4 TEST RESULTS")
            if is_windowed:
                print(f"(Windowed mode, window_size={pl_module.model.window_size})")
            print("=" * 60)
            
            if self.finger_idx is not None:
                y_true = y_true_all[:, self.finger_idx]
                corr = corr_metric(y_true.flatten(), y_pred.flatten())
                print(f"{self.finger_names[self.finger_idx]}: {corr:.4f}")
            else:
                corrs = []
                for i, name in enumerate(self.finger_names):
                    corr = corr_metric(y_true_all[:, i], y_pred[:, i])
                    corrs.append(corr)
                    print(f"  {name}: {corr:.4f}")
                print(f"\n  Average: {np.mean(corrs):.4f}")
            
            print("=" * 60)

In [ ]:
# =============================================================================
# MODEL AND DATAMODULE INITIALIZATION
# =============================================================================

if MODEL_MODE == 'fingerflex':
    SAMPLE_LEN = 256
    finger_num = 5

    hp_autoencoder = dict(
        channels = [32, 32, 64, 64, 128, 128], 
        kernel_sizes=[7, 7, 5, 5, 5],
        strides=[2, 2, 2, 2, 2],
        dilation=[1, 1, 1, 1, 1],
        n_electrodes = CHANNELS_NUM,
        n_freqs = WAVELET_NUM,
        n_channels_out = finger_num
    )

    model = AutoEncoder1D(**hp_autoencoder).to("cuda:0")
    lightning_wrapper = BaseEcogFingerflexModel(model)
    
    dm = EcogFingerflexDatamodule(
        sample_len=SAMPLE_LEN, 
        add_name="",
        model_mode='fingerflex'
    )
    
    print("=" * 60)
    print("FINGERFLEX MODEL (U-Net)")
    print("=" * 60)
    summary(model, torch.zeros(4, CHANNELS_NUM, WAVELET_NUM, SAMPLE_LEN).to("cuda:0"), show_input=False)

elif MODEL_MODE == 'bc4d4_windowed':
    # ==========================================================================
    # BC4D4 WINDOWED - Re-interpreted from paper
    # ==========================================================================
    n_outputs = 1 if BC4D4_FINGER_IDX is not None else 5
    finger_num = 1 if BC4D4_FINGER_IDX is not None else 5
    
    model = BC4D4Windowed(
        num_electrodes=CHANNELS_NUM,
        window_size=BC4D4_WINDOW_SIZE,
        activation=BC4D4_ACTIVATION,
        dropout_rate=0.1,
        n_outputs=n_outputs
    ).to("cuda:0")
    
    lightning_wrapper = BC4D4LightningModel(model, lr=1e-3)
    
    # Use the bc4d4_iso preprocessed data (has correct Isolation Forest)
    dm = EcogFingerflexDatamodule(
        sample_len=1,  # Not used
        add_name="_bc4d4_iso",  # Use Isolation Forest preprocessed data
        model_mode='bc4d4_windowed',
        finger_idx=BC4D4_FINGER_IDX,
        window_size=BC4D4_WINDOW_SIZE,
        stride=1,
        batch_size=64
    )
    
    print("=" * 60)
    print(f"BC4D4 WINDOWED MODEL - {BC4D4_ACTIVATION.upper()} activation")
    print("=" * 60)
    print(f"Window size: {BC4D4_WINDOW_SIZE} time steps")
    print(f"Number of electrodes: {CHANNELS_NUM}")
    print(f"Number of outputs: {n_outputs}")
    if BC4D4_FINGER_IDX is not None:
        print(f"Training finger: {FINGER_NAMES[BC4D4_FINGER_IDX]}")
    print(f"Total parameters: {model.count_parameters():,}")
    print(f"Flatten size: {model.flatten_size} (should be 256 to match paper)")
    
    # Verify parameter count matches paper
    fc1_params = model.flatten_size * 1024 + 1024
    print(f"FC1 params: {fc1_params:,} (paper says 262,144)")
    
    # Test forward pass
    test_input = torch.zeros(4, BC4D4_WINDOW_SIZE, CHANNELS_NUM).to("cuda:0")
    test_output = model(test_input)
    print(f"Input shape: {test_input.shape} -> Output shape: {test_output.shape}")

elif MODEL_MODE in ['bc4d4', 'bc4d4_iso']:
    # Original single time point BC4D4
    n_outputs = 1 if BC4D4_FINGER_IDX is not None else 5
    finger_num = 1 if BC4D4_FINGER_IDX is not None else 5
    
    data_suffix = "_bc4d4" if MODEL_MODE == 'bc4d4' else "_bc4d4_iso"
    
    model = BC4D4(
        num_features=CHANNELS_NUM,
        activation=BC4D4_ACTIVATION,
        dropout_rate=0.1,
        n_outputs=n_outputs
    ).to("cuda:0")
    
    lightning_wrapper = BC4D4LightningModel(model, lr=1e-3)
    
    dm = EcogFingerflexDatamodule(
        sample_len=1,
        add_name=data_suffix,
        model_mode='bc4d4',
        finger_idx=BC4D4_FINGER_IDX,
        batch_size=64
    )
    
    print("=" * 60)
    print(f"BC4D4 MODEL (Single Time Point) - {BC4D4_ACTIVATION.upper()}")
    print("=" * 60)
    print(f"WARNING: Single time point prediction typically doesn't generalize well")
    print(f"Total parameters: {model.count_parameters():,}")

else:
    raise ValueError(f"Unknown MODEL_MODE: {MODEL_MODE}")

In [ ]:
SAVE_PATH = f"{pathlib.Path().resolve()}/data"

def load_data(ecog_data_path, fingerflex_data_path):
    ecog_data = np.load(ecog_data_path)
    fingerflex_data = np.load(fingerflex_data_path)
    return ecog_data, fingerflex_data

# Load validation data based on model mode
if MODEL_MODE == 'fingerflex':
    ecog_data_val, fingerflex_data_val = load_data(
        f"{SAVE_PATH}/val/ecog_data.npy",
        f"{SAVE_PATH}/val/fingerflex_data.npy"
    )
elif MODEL_MODE == 'bc4d4':
    ecog_data_val, fingerflex_data_val = load_data(
        f"{SAVE_PATH}/val/ecog_data_bc4d4.npy",
        f"{SAVE_PATH}/val/fingerflex_data_bc4d4.npy"
    )
elif MODEL_MODE in ['bc4d4_iso', 'bc4d4_windowed']:
    # Both use the Isolation Forest preprocessed data
    ecog_data_val, fingerflex_data_val = load_data(
        f"{SAVE_PATH}/val/ecog_data_bc4d4_iso.npy",
        f"{SAVE_PATH}/val/fingerflex_data_bc4d4_iso.npy"
    )

print(f"Validation ECoG shape: {ecog_data_val.shape}")
print(f"Validation finger shape: {fingerflex_data_val.shape}")

In [ ]:
### TRAINING / TESTING ###
from pytorch_lightning.plugins.environments import LightningEnvironment

if TYPE == "train":
    wandb.init(project="BCI_comp")
    wandb_logger = WandbLogger()

    # Create checkpoint filename with finger name for BC4D4
    if MODEL_MODE in ['bc4d4', 'bc4d4_iso', 'bc4d4_windowed'] and BC4D4_FINGER_IDX is not None:
        ckpt_filename = f"{MODEL_MODE}-{FINGER_NAMES[BC4D4_FINGER_IDX].lower()}-{{epoch:02d}}-{{corr_mean_val:.4f}}"
    else:
        ckpt_filename = f"{MODEL_MODE}-model-{{epoch:02d}}-{{corr_mean_val:.4f}}"

    checkpoint_callback = ModelCheckpoint(
        save_top_k=2,
        monitor="corr_mean_val",
        mode="max",
        dirpath="checkpoints",
        filename=ckpt_filename,
    )

    # Select appropriate validation callback based on model
    if MODEL_MODE == 'fingerflex':
        val_callback = ValidationCallback(ecog_data_val, fingerflex_data_val, finger_num)
    elif MODEL_MODE in ['bc4d4', 'bc4d4_iso', 'bc4d4_windowed']:
        # BC4D4ValidationCallback works for all BC4D4 variants
        # For windowed, we pass the raw data and the callback handles windowing internally
        val_callback = BC4D4ValidationCallback(ecog_data_val, fingerflex_data_val, BC4D4_FINGER_IDX)

    trainer = Trainer(
        accelerator='gpu',
        devices=1,
        max_epochs=100 if MODEL_MODE in ['bc4d4', 'bc4d4_iso', 'bc4d4_windowed'] else 20,
        logger=wandb_logger,
        plugins=[LightningEnvironment()],
        callbacks=[val_callback, checkpoint_callback]
    )

    print(f"\n{'='*60}")
    print(f"Starting training: {MODEL_MODE.upper()} model")
    if MODEL_MODE in ['bc4d4', 'bc4d4_iso', 'bc4d4_windowed'] and BC4D4_FINGER_IDX is not None:
        print(f"Finger: {FINGER_NAMES[BC4D4_FINGER_IDX]}")
    if MODEL_MODE == 'bc4d4_iso':
        print("Using CORRECT Isolation Forest preprocessing")
    if MODEL_MODE == 'bc4d4_windowed':
        print(f"Using temporal window of {BC4D4_WINDOW_SIZE} time steps")
    print(f"{'='*60}")

    trainer.fit(lightning_wrapper, dm)
    wandb.finish()

elif TYPE == "test":
    ### TEST MODE ###
    if MODEL_MODE == 'fingerflex':
        trained_model = BaseEcogFingerflexModel.load_from_checkpoint(
            checkpoint_path=model_to_test,
            model=AutoEncoder1D(**hp_autoencoder)
        )
        trained_model = trained_model.cuda()

        test_callback = TestCallback(ecog_data_val, fingerflex_data_val, finger_num)
        test_callback.test(trained_model)

    elif MODEL_MODE in ['bc4d4', 'bc4d4_iso']:
        # For BC4D4, you need to specify the checkpoint path
        bc4d4_model = BC4D4(
            num_features=CHANNELS_NUM,
            activation=BC4D4_ACTIVATION,
            n_outputs=1 if BC4D4_FINGER_IDX is not None else 5
        )

        # Load checkpoint if exists
        if BC4D4_FINGER_IDX is not None:
            bc4d4_checkpoint = f"{pathlib.Path().resolve()}/checkpoints/{MODEL_MODE}-{FINGER_NAMES[BC4D4_FINGER_IDX].lower()}-best.ckpt"
        else:
            bc4d4_checkpoint = f"{pathlib.Path().resolve()}/checkpoints/{MODEL_MODE}-model-best.ckpt"

        if pathlib.Path(bc4d4_checkpoint).exists():
            trained_model = BC4D4LightningModel.load_from_checkpoint(
                checkpoint_path=bc4d4_checkpoint,
                model=bc4d4_model
            )
            trained_model = trained_model.cuda()

            test_callback = BC4D4TestCallback(ecog_data_val, fingerflex_data_val, BC4D4_FINGER_IDX)
            test_callback.test(trained_model)
        else:
            print(f"BC4D4 checkpoint not found at: {bc4d4_checkpoint}")
            print("Please train the model first or update the checkpoint path.")

    elif MODEL_MODE == 'bc4d4_windowed':
        # BC4D4 Windowed model testing
        bc4d4_model = BC4D4Windowed(
            num_electrodes=CHANNELS_NUM,
            window_size=BC4D4_WINDOW_SIZE,
            activation=BC4D4_ACTIVATION,
            n_outputs=1 if BC4D4_FINGER_IDX is not None else 5
        )

        if BC4D4_FINGER_IDX is not None:
            bc4d4_checkpoint = f"{pathlib.Path().resolve()}/checkpoints/{MODEL_MODE}-{FINGER_NAMES[BC4D4_FINGER_IDX].lower()}-best.ckpt"
        else:
            bc4d4_checkpoint = f"{pathlib.Path().resolve()}/checkpoints/{MODEL_MODE}-model-best.ckpt"

        if pathlib.Path(bc4d4_checkpoint).exists():
            trained_model = BC4D4LightningModel.load_from_checkpoint(
                checkpoint_path=bc4d4_checkpoint,
                model=bc4d4_model
            )
            trained_model = trained_model.cuda()

            test_callback = BC4D4TestCallback(ecog_data_val, fingerflex_data_val, BC4D4_FINGER_IDX)
            test_callback.test(trained_model)
        else:
            print(f"BC4D4 Windowed checkpoint not found at: {bc4d4_checkpoint}")
            print("Please train the model first or update the checkpoint path.")